# 02 – Train Pair-TCN + MIL phát hiện ĐÁNH NHAU

Sinh tự động bởi `realtime/tools/build_kaggle_notebook.py` – **không sửa tay**, sửa ở repo rồi sinh lại.

**Chuẩn bị:** Add Input → *Notebook Output* của `01_extract_pose` · Settings → Accelerator **GPU T4** ·
Settings → **Internet on** (để cài `onnx` nếu thiếu).

**Chạy:** để `QUICK = True` chạy thử (200 clip, 3 epoch, vài phút) → không lỗi thì đổi `QUICK = False`,
*Save Version → Save & Run All* để train đầy đủ (40 epoch, 3 thí nghiệm).
Ở QUICK chỉ kiểm tra **chạy không lỗi**: F1/accuracy val dao động mạnh là bình thường (rất ít bước train,
BatchNorm chưa ổn định) – chỉ đọc kết quả của bản train đầy đủ.

**Đầu ra (/kaggle/working):** `best_<thí nghiệm>.pt`, `pair_tcn.onnx`, `fight_config.json`,
`results.csv`, `instance_coverage.csv`, `clip_stats.csv`.

## 0. Thư viện xuất ONNX (cần bật Internet: Settings → Internet on)

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("onnx") is None:
    print("Chua co onnx -> pip install onnx onnxscript ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript"])
    assert r.returncode == 0, "pip install loi -> bat Internet: Settings -> Internet on, roi chay lai o nay"
import onnx
print("onnx", onnx.__version__)

## 1. Code dùng chung với realtime (`shared/sampling.py`, `shared/features.py`)

In [ ]:
%%writefile sampling.py
"""Lấy mẫu chuỗi frame về đúng FPS mục tiêu theo THỜI GIAN.

File này DÙNG CHUNG giữa realtime và notebook Kaggle -> chỉ dùng thư viện chuẩn.
Nguồn 30 FPS -> giữ ~1/2 số frame; nguồn 22 FPS (RLVS) -> giữ ~2/3; nguồn < fps -> giữ hết.
"""


class FpsSampler:
    def __init__(self, fps=15, tolerance=0.25):
        self.period = 1.0 / fps
        self.tol = tolerance * self.period     # bù rung thời gian (jitter) của camera
        self.next_t = None

    def reset(self):
        self.next_t = None

    def tick(self, t):
        """True nếu frame tại thời điểm t (giây) được lấy mẫu."""
        if self.next_t is None or t - self.next_t > self.period:   # frame đầu hoặc bị tụt lại xa
            self.next_t = t + self.period
            return True
        if t >= self.next_t - self.tol:
            self.next_t += self.period
            return True
        return False


def sample_indices(times, fps=15):
    """Offline: trả về chỉ số các frame được giữ, vd times = [i / fps_goc for i in range(n)]."""
    s = FpsSampler(fps)
    return [i for i, t in enumerate(times) if s.tick(t)]

In [ ]:
%%writefile features.py
"""Đặc trưng CẶP NGƯỜI cho mô hình ĐÁNH NHAU (Pair-TCN) – CLAUDE.md mục 7.1.

DÙNG CHUNG giữa notebook Kaggle (train) và realtime (detectors/fight.py) -> chỉ dùng numpy.
Mọi thứ tính trên trục thời gian ĐÃ LẤY MẪU về 15 FPS: frame mẫu thứ k ứng với thời điểm ≈ k / 15 s,
nên không đặc trưng nào phụ thuộc FPS gốc hay số frame của video.

Dạng dữ liệu:
- tracks (theo thời gian thật):  {tid: [(t_giây, box_px(4,), kpts_px(17, 3)), ...]}
- tracks_k (đã lấy mẫu 15 FPS):  {tid: {k: (box_px(4,), kpts_px(17, 3))}}
- seq (1 người trong 1 cửa sổ):  {"box": (..., W, 4), "kpts": (..., W, 17, 3), "valid": (..., W) bool,
                                  "present": (..., W) bool}   (valid = có dữ liệu sau khi lấp frame thiếu)
"""
import numpy as np

try:
    from shared.sampling import sample_indices          # realtime (chạy từ thư mục realtime/)
except ImportError:
    from sampling import sample_indices                 # Kaggle (%%writefile sampling.py)

FPS = 15
WINDOW = 32                 # frame mẫu / cửa sổ (~2.1 s)
STRIDE = 8
FEAT_DIM = 180
KPT_CONF = 0.3              # keypoint conf < ngưỡng -> toạ độ 0, conf 0
MAX_FILL = 3                # lấp (nội suy) khoảng trống ≤ N frame mẫu
MIN_PRESENCE = 0.6          # mỗi người phải có mặt ≥ 60% frame của cửa sổ
NEAR_RATIO = 1.5            # ≥ 1 frame có khoảng cách tâm ≤ 1.5 × chiều cao TB
MAX_PAIRS = 6               # tối đa N cặp gần nhau nhất / cửa sổ
LINK_GAP_SEC = 0.5          # nối tracklet: B bắt đầu ≤ 0.5 s sau khi A kết thúc...
LINK_DIST = 0.5             # ...và tâm box cách nhau < 0.5 × chiều cao box
FAR = 3.0                   # khoảng cách khi thiếu dữ liệu (và giới hạn trên), đơn vị chiều cao người
VEL_CLIP = 1.0              # chặn vận tốc (chiều cao / frame mẫu) – bỏ nhảy do đổi ID

HEAD = [0, 1, 2, 3, 4]
TORSO = [5, 6, 11, 12]
L_WR, R_WR, L_HIP, R_HIP = 9, 10, 11, 12
FLIP_KPTS = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]   # hoán đổi trái/phải

# Bố cục 180 đặc trưng mỗi frame
SL_XY = slice(0, 68)        # A: 17×(x, y) -> 0..33, B -> 34..67  (gốc = trung điểm hông A, B; chia thang đo)
SL_CONF = slice(68, 102)    # A: 68..84, B: 85..101
SL_VEL = slice(102, 170)    # A: 102..135, B: 136..169 (hiệu 2 frame mẫu liên tiếp, đã chuẩn hoá)
SL_WRIST = slice(170, 178)  # A_Ltay→B_đầu, A_Ltay→B_thân, A_Rtay→B_đầu, A_Rtay→B_thân, rồi B→A tương tự
SL_DIST = slice(178, 180)   # khoảng cách tâm A–B, tốc độ thay đổi của nó


# ---------------------------------------------------------------- dữ liệu offline (npz)

def tracks_from_npz(npz):
    """npz của notebook 01_extract_pose -> {tid: [(t, box_px, kpts_px)]} ở FPS GỐC.
    Toạ độ trong npz chuẩn hoá 0..1 theo W và H riêng -> nhân lại về pixel (tránh méo tỉ lệ). Bỏ tid -1."""
    data = np.asarray(npz["data"], np.float32).reshape(-1, 57)
    fps, W, H = float(npz["fps"]), float(npz["width"]), float(npz["height"])
    tracks = {}
    data = data[data[:, 1] >= 0]
    for row in data[np.lexsort((data[:, 0], data[:, 1]))]:
        box = row[2:6] * np.array([W, H, W, H], np.float32)
        k = row[6:].reshape(17, 3).copy()
        k[:, 0] *= W
        k[:, 1] *= H
        tracks.setdefault(int(row[1]), []).append((float(row[0]) / fps, box, k))
    return tracks


def clip_times(npz):
    """Thời điểm của mọi frame trong clip (kể cả frame không có ai) – lưới để lấy mẫu."""
    return np.arange(int(npz["n_frames"])) / float(npz["fps"])


def sample_tracks(tracks, fps=FPS, times=None):
    """Lấy mẫu về fps theo THỜI GIAN bằng logic FpsSampler trên lưới thời gian CHUNG của clip
    (mọi người cùng các frame mẫu). Frame mẫu tại thời điểm t có chỉ số k = round(t × fps).
    times: thời điểm mọi frame của clip (clip_times); None = hợp các thời điểm có người."""
    if times is None:
        times = sorted({t for tr in tracks.values() for t, _, _ in tr})
    times = np.asarray(times, float)
    keep = {round(float(times[i]), 6) for i in sample_indices(list(times), fps)}
    out = {}
    for tid, tr in tracks.items():
        d = {}
        for t, box, k in tr:
            if round(t, 6) in keep:
                d.setdefault(int(round(t * fps)), (np.asarray(box, np.float32), np.asarray(k, np.float32)))
        if d:
            out[tid] = d
    return out


def _center(box):
    return np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])


def link_tracklets(tracks_k, fps=FPS, max_gap_sec=LINK_GAP_SEC, max_dist=LINK_DIST):
    """Nối tracklet (offline, vì ByteTrack persist=False hay đổi ID): track A kết thúc, track B bắt đầu
    sau đó ≤ max_gap_sec giây và tâm box cách nhau < max_dist × chiều cao box của A -> gộp B vào A.
    Mỗi B chọn A gần nhất. Trả về dict mới {tid: {k: (box, kpts)}}."""
    merged = {tid: dict(d) for tid, d in tracks_k.items()}
    heads = {}                                   # tid chuỗi -> (k cuối, box cuối)
    for s, e, tid in sorted((min(d), max(d), tid) for tid, d in tracks_k.items()):
        first_box = merged[tid][s][0]
        best = None
        for a, (ae, abox) in heads.items():
            gap = s - ae
            if 0 < gap <= max_gap_sec * fps + 1e-9:
                dist = np.linalg.norm(_center(first_box) - _center(abox))
                if dist < max_dist * (abox[3] - abox[1]) and (best is None or dist < best[0]):
                    best = (dist, a)
        if best is None:
            heads[tid] = (e, merged[tid][e][0])
        else:
            a = best[1]
            merged[a].update(merged.pop(tid))
            heads[a] = (e, merged[a][e][0])
    return merged


# ---------------------------------------------------------------- cửa sổ, cặp

def fill_gaps(box, kpts, present, max_fill=MAX_FILL):
    """Lấp frame thiếu của 1 người trong cửa sổ (sửa tại chỗ): khoảng trống ≤ max_fill frame giữa 2 frame
    có dữ liệu -> nội suy tuyến tính (conf = min 2 đầu); còn lại để 0, conf 0. Trả về mask valid."""
    valid = present.copy()
    idx = np.flatnonzero(present)
    for i, j in zip(idx[:-1], idx[1:]):
        if 1 < j - i <= max_fill + 1:
            for m in range(i + 1, j):
                w = (m - i) / (j - i)
                box[m] = (1 - w) * box[i] + w * box[j]
                kpts[m, :, :2] = (1 - w) * kpts[i, :, :2] + w * kpts[j, :, :2]
                kpts[m, :, 2] = np.minimum(kpts[i, :, 2], kpts[j, :, 2])
                valid[m] = True
    return valid


def seq_from_track(track_k, start, window=WINDOW):
    """1 người (dict k -> (box, kpts)) trong cửa sổ [start, start + window) -> seq (đã lấp frame thiếu)."""
    box = np.zeros((window, 4), np.float32)
    kpts = np.zeros((window, 17, 3), np.float32)
    present = np.zeros(window, bool)
    for k, (b, kp) in track_k.items():
        i = k - start
        if 0 <= i < window:
            box[i], kpts[i], present[i] = b, kp, True
    valid = fill_gaps(box, kpts, present)
    return {"box": box, "kpts": kpts, "valid": valid, "present": present}


def candidate_pairs(seqs, min_presence=MIN_PRESENCE, near=NEAR_RATIO, max_pairs=MAX_PAIRS):
    """Cặp ứng viên trong cửa sổ: cả 2 có mặt ≥ min_presence số frame; có ≥ 1 frame khoảng cách tâm box
    ≤ near × chiều cao TB của 2 người. Trả về ≤ max_pairs cặp (idA < idB) gần nhau nhất."""
    ids = sorted(t for t, s in seqs.items() if s["present"].mean() >= min_presence)
    scored = []
    for i, a in enumerate(ids):
        A = seqs[a]
        for b in ids[i + 1:]:
            B = seqs[b]
            both = A["valid"] & B["valid"]
            if not both.any():
                continue
            ca = (A["box"][both, :2] + A["box"][both, 2:]) / 2
            cb = (B["box"][both, :2] + B["box"][both, 2:]) / 2
            h = ((A["box"][both, 3] - A["box"][both, 1]) + (B["box"][both, 3] - B["box"][both, 1])) / 2
            d = (np.linalg.norm(ca - cb, axis=1) / np.maximum(h, 1.0)).min()
            if d <= near:
                scored.append((d, a, b))
    return [(a, b) for _, a, b in sorted(scored)[:max_pairs]]


def build_pair_instances(tracks_k, window=WINDOW, stride=STRIDE, starts=None):
    """-> list[(idA, idB, start_k, seqA, seqB)] cho mọi cửa sổ (hoặc các cửa sổ bắt đầu tại `starts`,
    vd realtime chỉ xét cửa sổ mới nhất). Clip ngắn hơn 1 cửa sổ -> 1 cửa sổ, phần thiếu coi như vắng."""
    if not tracks_k:
        return []
    if starts is None:
        k0 = min(min(d) for d in tracks_k.values())
        k1 = max(max(d) for d in tracks_k.values())
        starts = list(range(k0, k1 - window + 2, stride)) or [k0]
    out = []
    for s in starts:
        seqs = {tid: seq_from_track(d, s, window) for tid, d in tracks_k.items()
                if sum(s <= k < s + window for k in d) >= MIN_PRESENCE * window}
        for a, b in candidate_pairs(seqs):
            out.append((a, b, s, seqs[a], seqs[b]))
    return out


def build_pair_windows(tracks_k, window=WINDOW, stride=STRIDE, fps=FPS, starts=None):
    """-> list[(idA, idB, start_t, feats(window, 180))] (CLAUDE.md 7.1 mục 8)."""
    return [(a, b, s / fps, pair_features(A, B))
            for a, b, s, A, B in build_pair_instances(tracks_k, window, stride, starts)]


def clip_instances(npz, fps=FPS, window=WINDOW, stride=STRIDE):
    """Toàn bộ pipeline offline cho 1 clip: npz -> theo track -> lấy mẫu 15 FPS -> nối tracklet -> cặp/cửa sổ.
    Trả về (instances [(idA, idB, start_k, seqA, seqB)], tracks_k)."""
    tracks_k = link_tracklets(sample_tracks(tracks_from_npz(npz), fps, clip_times(npz)), fps)
    return build_pair_instances(tracks_k, window, stride), tracks_k


# ---------------------------------------------------------------- đặc trưng

def _masked_mean(pts, ok):
    """pts (..., n, 2), ok (..., n) -> (trung bình các điểm ok (..., 2), có ít nhất 1 điểm (...))"""
    n = ok.sum(-1)
    m = (pts * ok[..., None]).sum(-2) / np.maximum(n, 1)[..., None]
    return m, n > 0


def _person(seq):
    k = np.asarray(seq["kpts"], np.float32)
    box = np.asarray(seq["box"], np.float32)
    valid = np.asarray(seq["valid"], bool)
    ok = (k[..., 2] >= KPT_CONF) & valid[..., None]                 # (..., W, 17)
    center = (box[..., :2] + box[..., 2:]) / 2
    hip, has_hip = _masked_mean(k[..., [L_HIP, R_HIP], :2], ok[..., [L_HIP, R_HIP]])
    hip = np.where(has_hip[..., None], hip, center)                 # thiếu hông -> tâm box
    height = box[..., 3] - box[..., 1]
    return k, ok, valid, center, hip, height


def pair_features(seqA, seqB):
    """Đặc trưng mỗi frame của cặp (A, B) -> (..., W, 180) float32 (hỗ trợ batch ở các chiều đầu).
    Gốc toạ độ = trung điểm tâm hông A và B (thiếu hông -> tâm box) ở TỪNG frame; thang đo S = chiều cao box
    trung bình của 2 người trong cửa sổ. Vận tốc tính với gốc cố định (trung bình cửa sổ) để giữ chuyển động."""
    kA, okA, vA, cA, hipA, hA = _person(seqA)
    kB, okB, vB, cB, hipB, hB = _person(seqB)

    def mean_h(h, v):
        return (h * v).sum(-1) / np.maximum(v.sum(-1), 1)
    nA, nB = vA.any(-1), vB.any(-1)
    S = np.where(nA & nB, (mean_h(hA, vA) + mean_h(hB, vB)) / 2, np.where(nA, mean_h(hA, vA), mean_h(hB, vB)))
    S = np.maximum(S, 1.0)                                          # (...)
    S1, S3 = S[..., None], S[..., None, None, None]

    both = vA & vB
    org = np.where(both[..., None], (hipA + hipB) / 2,
                   np.where(vA[..., None], hipA, np.where(vB[..., None], hipB, 0.0)))   # (..., W, 2)
    anyv = vA | vB
    O = (org * anyv[..., None]).sum(-2) / np.maximum(anyv.sum(-1), 1)[..., None]     # (..., 2)

    def xy(k, ok):
        return np.where(ok[..., None], (k[..., :2] - org[..., None, :]) / S3, 0.0)

    def vel(k, ok):
        q = (k[..., :2] - O[..., None, None, :]) / S3
        v = np.zeros_like(q)
        m = ok[..., 1:, :] & ok[..., :-1, :]
        v[..., 1:, :, :] = np.where(m[..., None], q[..., 1:, :, :] - q[..., :-1, :, :], 0.0)
        return np.clip(v, -VEL_CLIP, VEL_CLIP)

    def dist(p, okp, target, has):
        d = np.linalg.norm(p - target, axis=-1) / S1
        return np.where(okp & has, np.minimum(d, FAR), FAR)

    def targets(k, ok, center, v):
        head, has_head = _masked_mean(k[..., HEAD, :2], ok[..., HEAD])
        torso, has_torso = _masked_mean(k[..., TORSO, :2], ok[..., TORSO])
        torso = np.where(has_torso[..., None], torso, center)
        return head, has_head, torso, has_torso | v

    hdB, hhB, toB, htB = targets(kB, okB, cB, vB)
    hdA, hhA, toA, htA = targets(kA, okA, cA, vA)
    wrist = []
    for k, ok, hd, hh, to, ht in ((kA, okA, hdB, hhB, toB, htB), (kB, okB, hdA, hhA, toA, htA)):
        for w in (L_WR, R_WR):
            wrist += [dist(k[..., w, :2], ok[..., w], hd, hh), dist(k[..., w, :2], ok[..., w], to, ht)]

    dc = np.where(both, np.minimum(np.linalg.norm(cA - cB, axis=-1) / S1, FAR), FAR)
    rate = np.zeros_like(dc)
    rate[..., 1:] = np.where(both[..., 1:] & both[..., :-1], dc[..., 1:] - dc[..., :-1], 0.0)

    W = kA.shape[-3]
    lead = kA.shape[:-3]
    flat = lambda a: a.reshape(*lead, W, -1)                       # noqa: E731
    feats = np.concatenate([
        flat(xy(kA, okA)), flat(xy(kB, okB)),
        np.where(okA, kA[..., 2], 0.0), np.where(okB, kB[..., 2], 0.0),
        flat(vel(kA, okA)), flat(vel(kB, okB)),
        np.stack(wrist, -1), dc[..., None], rate[..., None]], axis=-1)
    return feats.astype(np.float32)


# ---------------------------------------------------------------- đối xứng (kiểm tra / tăng cường)

def _swap_perm():
    a, b = np.arange(0, 34), np.arange(34, 68)
    ca, cb = np.arange(68, 85), np.arange(85, 102)
    va, vb = np.arange(102, 136), np.arange(136, 170)
    wa, wb = np.arange(170, 174), np.arange(174, 178)
    return np.concatenate([b, a, cb, ca, vb, va, wb, wa, [178, 179]])


def _flip_perm_sign():
    perm, sign = np.arange(FEAT_DIM), np.ones(FEAT_DIM, np.float32)
    for base in (0, 34, 102, 136):                                   # khối (x, y) × 17
        for j, fj in enumerate(FLIP_KPTS):
            perm[base + 2 * j] = base + 2 * fj
            perm[base + 2 * j + 1] = base + 2 * fj + 1
            sign[base + 2 * j] = -1.0                                # x -> -x
    for base in (68, 85):                                            # conf × 17
        for j, fj in enumerate(FLIP_KPTS):
            perm[base + j] = base + fj
    for base in (170, 174):                                          # (Ltay->đầu, Ltay->thân, Rtay->đầu, Rtay->thân)
        perm[base:base + 4] = [base + 2, base + 3, base, base + 1]
    return perm, sign


SWAP_PERM = _swap_perm()
FLIP_PERM, FLIP_SIGN = _flip_perm_sign()


def swap_ab(feats):
    """Đặc trưng của cặp (B, A) từ đặc trưng cặp (A, B)."""
    return feats[..., SWAP_PERM]


def flip_h(feats):
    """Đặc trưng của cặp đã lật ngang (x -> -x, hoán đổi khớp trái/phải)."""
    return feats[..., FLIP_PERM] * FLIP_SIGN

## 2. Cấu hình

In [ ]:
QUICK = True            # True: chạy thử nhanh. Chạy ổn rồi đổi False để train đầy đủ.
QUICK_CLIPS = 200       # số clip khi QUICK (chia đều dataset × split × nhãn)
EPOCHS = 3 if QUICK else 40
BATCH_BAGS = 16         # số bag (clip) / batch
LR, WEIGHT_DECAY = 1e-3, 1e-4
MAX_INST_TRAIN = 64     # tối đa instance / bag khi train (clip RLVS dài có thể > 200 instance)
THRESHOLD = 0.6         # ghi vào fight_config.json (ngưỡng realtime, CLAUDE.md 7.4)
SEED = 42
EXPERIMENTS = [("RWF", ["RWF"]), ("RLVS", ["RLVS"]), ("GOP", ["RWF", "RLVS"])]   # (tên, dataset dùng để train)
EXPORT_FROM = "GOP"     # thí nghiệm dùng để xuất pair_tcn.onnx

import os, sys, glob, json, time, math, random, importlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

INPUT = os.environ.get("HTTM_INPUT", "/kaggle/input")
WORK = os.environ.get("HTTM_WORK", "/kaggle/working")
N_WORKERS = int(os.environ.get("HTTM_WORKERS", os.cpu_count() or 2))
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, os.getcwd())
import sampling, features as F
importlib.reload(sampling); importlib.reload(F)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "KHONG CO GPU (Settings -> Accelerator)")
print("QUICK =", QUICK, "| EPOCHS =", EPOCHS, "| CPU workers =", N_WORKERS)
print("features: FPS", F.FPS, "WINDOW", F.WINDOW, "STRIDE", F.STRIDE, "FEAT_DIM", F.FEAT_DIM)

## 3. Tìm dữ liệu (đầu ra của `01_extract_pose`)

In [ ]:
metas = sorted(glob.glob(os.path.join(INPUT, "**", "meta.csv"), recursive=True))
print("meta.csv tim thay:", metas)
assert metas, "Khong thay meta.csv trong /kaggle/input -> Add Input -> Notebook Output cua 01_extract_pose"
META = metas[0]
if len(metas) > 1:
    print("!! Co nhieu meta.csv, dung file dau tien:", META)
POSES = os.path.join(os.path.dirname(META), "poses")
if not os.path.isdir(POSES):
    dirs = [d for d in glob.glob(os.path.join(INPUT, "**", "poses"), recursive=True) if os.path.isdir(d)]
    assert dirs, "Khong thay thu muc poses/"
    POSES = dirs[0]
print("META  =", META)
print("POSES =", POSES, "|", len(glob.glob(os.path.join(POSES, "*.npz"))), "file .npz")

In [ ]:
# Định dạng của 01_extract_pose: dataset ∈ {rwf2000, rlvs}; label 0/1 (1 = Fight/Violence);
# split ∈ {train, val}; file = "{dataset}__{split}__{tên}__{label}.npz" trong poses/
COLS = ["file", "dataset", "label", "split", "src", "n_frames", "max_persons", "n_long_tracks", "fps", "has_pair"]
DS_NAME = {"rwf2000": "RWF", "rlvs": "RLVS"}

meta = pd.read_csv(META)
print("Cot:", meta.columns.tolist())
display(meta.head(3))

def check(bad, msg):
    if bad.any():
        display(meta.loc[bad, ["file", "dataset", "label", "split"]].head(10))
        raise ValueError(f"{msg}: {int(bad.sum())} dong (xem bang tren)")

missing_cols = [c for c in COLS if c not in meta.columns]
assert not missing_cols, f"meta.csv thieu cot {missing_cols}"
check(~meta["dataset"].isin(DS_NAME), f"dataset phai thuoc {sorted(DS_NAME)}")
check(~meta["split"].isin(["train", "val"]), "split phai la 'train' hoac 'val'")
label_num = pd.to_numeric(meta["label"], errors="coerce")
check(~label_num.isin([0, 1]), "label phai la so nguyen 0/1")
def name_ok(f, d, s, l):
    p = os.path.basename(str(f))[:-4].split("__") if str(f).endswith(".npz") else []
    return len(p) >= 4 and p[0] == d and p[1] == s and p[-1] == str(int(l))
expect_ok = np.array([name_ok(*r) for r in zip(meta["file"], meta["dataset"], meta["split"], label_num.fillna(-1))])
check(~expect_ok, "ten file khong khop dang {dataset}__{split}__{ten}__{label}.npz voi cac cot")

meta["ds"] = meta["dataset"].map(DS_NAME)
meta["y"] = label_num.astype(int)
meta["sp"] = meta["split"]
meta["path"] = meta["file"].map(lambda f: os.path.join(POSES, os.path.basename(str(f))))
ok = meta["path"].map(os.path.exists)
print(f"Thieu file npz: {(~ok).sum()} / {len(meta)}")
check(~ok, "khong thay file .npz trong poses/")
display(meta.groupby(["dataset", "split", "label"]).size().rename("so clip").reset_index())

if QUICK:
    per = max(1, QUICK_CLIPS // meta.groupby(["ds", "sp", "y"]).ngroups)
    take = [i for _, g in meta.groupby(["ds", "sp", "y"])
            for i in g.sample(min(len(g), per), random_state=SEED).index]
    meta = meta.loc[sorted(take)].reset_index(drop=True)
    print(f"QUICK: lay {len(meta)} clip")

NPZ_KEYS = ("data", "fps", "n_frames", "width", "height")

def load_npz(path):
    """Đọc npz và kiểm tra đúng định dạng (mục 6 CLAUDE.md); sai -> báo lỗi kèm tên file."""
    z = np.load(path)
    miss = [k for k in NPZ_KEYS if k not in z.files]
    if miss:
        raise ValueError(f"{os.path.basename(path)}: thieu khoa {miss} (co {z.files})")
    npz = {k: z[k] for k in NPZ_KEYS}
    d = npz["data"]
    if d.ndim != 2 or d.shape[1] != 57:
        if d.size == 0:                              # clip không có ai -> (0, 57)
            npz["data"] = d.reshape(0, 57).astype(np.float32)
        else:
            raise ValueError(f"{os.path.basename(path)}: data phai (N, 57), dang la {d.shape}")
    if not (float(npz["fps"]) > 0 and int(npz["n_frames"]) > 0 and int(npz["width"]) > 0 and int(npz["height"]) > 0):
        raise ValueError(f"{os.path.basename(path)}: fps/n_frames/width/height khong hop le")
    return npz

z = load_npz(meta.path.iloc[0])
print("npz mau:", {k: (v.shape, v.dtype) for k, v in z.items()})

## 4. Dựng instance (cặp × cửa sổ)
Mỗi clip: npz → theo track (pixel) → **lấy mẫu 15 FPS theo thời gian** → nối tracklet → cửa sổ 32 / bước 8 →
cặp ứng viên. Lưu **chuỗi khung xương thô** của mỗi instance (để tăng cường dữ liệu trước khi tính đặc trưng).

In [ ]:
from multiprocessing import Pool

def process(path):
    npz = load_npz(path)
    inst, tracks_k = F.clip_instances(npz)
    K = len(sampling.sample_indices(list(F.clip_times(npz)), F.FPS))   # số frame mẫu của clip
    st = dict(dur=float(npz["n_frames"]) / float(npz["fps"]), K=K,
              n_win=max(1, (K - F.WINDOW) // F.STRIDE + 1), n_inst=len(inst), n_tracks=len(tracks_k),
              n_win_pair=len({s for _, _, s, _, _ in inst}))
    W = F.WINDOW
    kp = np.zeros((len(inst), 2, W, 17, 3), np.float32)
    box = np.zeros((len(inst), 2, W, 4), np.float32)
    valid = np.zeros((len(inst), 2, W), bool)
    for i, (_, _, _, A, B) in enumerate(inst):
        for j, s in enumerate((A, B)):
            kp[i, j], box[i, j], valid[i, j] = s["kpts"], s["box"], s["valid"]
    return kp, box, valid, st

t0 = time.time()
paths = meta.path.tolist()
if N_WORKERS > 1:
    with Pool(N_WORKERS) as pool:
        res = pool.map(process, paths, chunksize=8)
else:
    res = [process(p) for p in paths]
BUILD_SEC = time.time() - t0

KP = np.concatenate([r[0] for r in res])
BOX = np.concatenate([r[1] for r in res])
VALID = np.concatenate([r[2] for r in res])
stats = pd.DataFrame([r[3] for r in res])
del res
meta = pd.concat([meta, stats], axis=1)
CLIP_OF = np.repeat(np.arange(len(meta)), meta.n_inst.values)            # instance -> clip
starts = np.concatenate([[0], np.cumsum(meta.n_inst.values)])
INST_OF = [np.arange(starts[c], starts[c + 1]) for c in range(len(meta))]  # clip -> các instance
LABEL = meta.y.values
meta["has_inst"] = meta.n_inst > 0
print(f"Dung {len(KP)} instance tu {len(meta)} clip trong {BUILD_SEC:.1f} s "
      f"({BUILD_SEC / len(meta) * 1000:.0f} ms/clip) | bo nho {KP.nbytes / 1e9:.2f} GB")

## 5. Kiểm tra dữ liệu

In [ ]:
def feats_of(idx):
    A = {"kpts": KP[idx, 0], "box": BOX[idx, 0], "valid": VALID[idx, 0]}
    B = {"kpts": KP[idx, 1], "box": BOX[idx, 1], "valid": VALID[idx, 1]}
    return F.pair_features(A, B)

lab = meta.y.map({0: "0 NonFight", 1: "1 Fight"})
print("== So clip theo dataset / split / nhan")
display(pd.crosstab([meta.ds, meta.sp], lab, margins=True))
print("== So clip co >= 1 instance cap")
display(pd.crosstab([meta.ds, meta.sp], lab, values=meta.has_inst, aggfunc="sum", margins=True))
print(f"== Tong so instance: {len(KP)} | thoi gian dung instance: {BUILD_SEC:.1f} s")
display(meta.groupby(["ds", "sp", lab])["n_inst"].agg(["sum", "mean", "median", "max"]).round(1))

### Kiểm tra lệch FPS (RLVS: Violence ~29.6 FPS, NonViolence ~24.9 FPS)
Mọi đặc trưng tính trên trục thời gian đã lấy mẫu về **15 FPS** → *FPS sau lấy mẫu* phải ≈ 15 ở mọi nhóm,
số cửa sổ/clip chỉ phụ thuộc **thời lượng**, vận tốc tính theo **chiều cao người / giây**.
Nếu FPS sau lấy mẫu khác nhau giữa 2 nhãn → mô hình có thể học "đường tắt" theo FPS → phải sửa trước khi train.

In [ ]:
meta["fps_sampled"] = meta.K / meta.dur
print("== FPS goc va FPS sau lay mau (phai ~15 o moi nhom)")
display(meta.groupby(["ds", lab])[["fps", "fps_sampled", "dur"]].agg(["mean", "min", "max"]).round(2))
print("== So cua so / clip (truc 15 FPS)")
display(meta.groupby(["ds", lab])["n_win"].describe().round(1))

rng = np.random.default_rng(SEED)
rows = []
for (ds, y), g in meta.groupby(["ds", "y"]):
    idx = np.concatenate([INST_OF[c] for c in g.index]) if g.n_inst.sum() else np.array([], int)
    if len(idx) == 0:
        continue
    idx = rng.choice(idx, min(len(idx), 3000), replace=False)
    f = feats_of(np.sort(idx))
    v = f[..., F.SL_VEL].reshape(*f.shape[:2], 34, 2)
    conf = f[..., F.SL_CONF] > 0
    speed = np.linalg.norm(v, axis=-1)[conf] * F.FPS                     # chiều cao người / giây
    wr = np.linalg.norm(v[:, :, [9, 10, 26, 27]], axis=-1)[conf[:, :, [9, 10, 26, 27]]] * F.FPS
    rows.append(dict(ds=ds, nhan=y, n_inst=len(idx), van_toc_TB=speed.mean(), van_toc_trung_vi=np.median(speed),
                     van_toc_co_tay_TB=wr.mean(), kc_tam_TB=np.median(f[..., 178][f[..., 178] < F.FAR])))
print("== Van toc khop trung binh theo nhan (chieu cao nguoi / giay, tinh tren truc 15 FPS)")
display(pd.DataFrame(rows).round(3))

### Giới hạn trên của recall: tỉ lệ clip ĐÁNH NHAU có ≥ 1 instance (clip không có cặp nào sẽ luôn bị đoán 0)

In [ ]:
cov = (meta[meta.y == 1].groupby(["ds", "sp"])
           .agg(clip_fight=("y", "size"), co_instance=("has_inst", "sum"), recall_toi_da=("has_inst", "mean"))
           .reset_index())
cov = pd.concat([cov, meta[meta.y == 1].groupby("sp").agg(clip_fight=("y", "size"), co_instance=("has_inst", "sum"),
                 recall_toi_da=("has_inst", "mean")).reset_index().assign(ds="GOP")], ignore_index=True)
display(cov.round(3))
cov.to_csv(os.path.join(WORK, "instance_coverage.csv"), index=False)
meta.drop(columns=["path"]).to_csv(os.path.join(WORK, "clip_stats.csv"), index=False)

## 6. Tăng cường dữ liệu, mô hình, MIL
Tăng cường trên **chuỗi khung xương thô** (rồi mới tính đặc trưng bằng `features.pair_features`):
co giãn thời gian 0.8–1.2 · hoán đổi A/B · lật ngang · nhiễu Gauss σ = 0.01 × chiều cao · xoá 10% khớp ·
30% xoá thân dưới (khớp 11–16, giả lập webcam thấy nửa người).

In [ ]:
def augment(kp, box, valid, rng):
    kp, box, valid = kp.copy(), box.copy(), valid.copy()
    n, _, W = valid.shape
    # co giãn thời gian quanh tâm cửa sổ (nội suy tuyến tính)
    r = rng.uniform(0.8, 1.2, n)
    c = (W - 1) / 2
    pos = np.clip(c + (np.arange(W)[None] - c) * r[:, None], 0, W - 1)
    i0 = np.floor(pos).astype(int)
    i1 = np.minimum(i0 + 1, W - 1)
    w = (pos - i0).astype(np.float32)
    rows = np.arange(n)[:, None]
    def interp(a):                                   # a (n, 2, W, ...) -> nội suy theo trục W
        t = np.moveaxis(a, 2, 1)
        ww = w.reshape(n, W, *([1] * (t.ndim - 2)))
        return np.moveaxis(t[rows, i0] * (1 - ww) + t[rows, i1] * ww, 1, 2)
    kp, box = interp(kp), interp(box)
    vt = np.moveaxis(valid, 2, 1)
    valid = np.moveaxis(vt[rows, i0] & vt[rows, i1], 1, 2)
    # hoán đổi A/B
    s = rng.random(n) < 0.5
    kp[s], box[s], valid[s] = kp[s][:, ::-1], box[s][:, ::-1], valid[s][:, ::-1]
    # lật ngang: x -> -x, đổi khớp trái/phải
    f = rng.random(n) < 0.5
    kp[f] = kp[f][..., F.FLIP_KPTS, :]
    kp[f, ..., 0] *= -1
    box[f] = np.stack([-box[f][..., 2], box[f][..., 1], -box[f][..., 0], box[f][..., 3]], -1)
    # nhiễu Gauss σ = 0.01 × chiều cao TB của cặp
    h = box[..., 3] - box[..., 1]
    S = np.maximum((h * valid).sum((1, 2)) / np.maximum(valid.sum((1, 2)), 1), 1.0)
    kp[..., :2] += rng.normal(0, 0.01, kp[..., :2].shape).astype(np.float32) * S[:, None, None, None, None]
    # xoá ngẫu nhiên 10% khớp; 30% xoá thân dưới
    kp[..., 2][rng.random(kp.shape[:-1]) < 0.1] = 0
    low = rng.random(n) < 0.3
    kp[low, :, :, 11:17, 2] = 0
    return kp, box, valid

class BagDS(torch.utils.data.Dataset):
    def __init__(self, clips, train, mean, std):
        self.clips, self.train, self.mean, self.std = list(clips), train, mean, std
    def __len__(self):
        return len(self.clips)
    def __getitem__(self, i):
        c = self.clips[i]
        idx = INST_OF[c]
        if self.train:
            rng = np.random.default_rng()
            if len(idx) > MAX_INST_TRAIN:
                idx = np.sort(rng.choice(idx, MAX_INST_TRAIN, replace=False))
            kp, box, valid = augment(KP[idx], BOX[idx], VALID[idx], rng)
            A = {"kpts": kp[:, 0], "box": box[:, 0], "valid": valid[:, 0]}
            B = {"kpts": kp[:, 1], "box": box[:, 1], "valid": valid[:, 1]}
            f = F.pair_features(A, B)
        else:
            f = feats_of(idx)
        f = (f - self.mean) / self.std
        return torch.from_numpy(f.astype(np.float32)), float(LABEL[c])

def collate(batch):
    return (torch.cat([f for f, _ in batch]), torch.tensor([len(f) for f, _ in batch]),
            torch.tensor([y for _, y in batch], dtype=torch.float32))

class Block(nn.Module):
    def __init__(self, c, d, p=0.2):
        super().__init__()
        self.conv = nn.Conv1d(c, c, 3, padding=d, dilation=d)
        self.bn = nn.BatchNorm1d(c)
        self.drop = nn.Dropout(p)
    def forward(self, x):
        return x + self.drop(torch.relu(self.bn(self.conv(x))))

class PairTCN(nn.Module):
    def __init__(self, d_in=F.FEAT_DIM, c=128):
        super().__init__()
        self.inp = nn.Conv1d(d_in, c, 1)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8)])
        self.head = nn.Sequential(nn.Linear(2 * c, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):                            # x (batch, 32, 180) -> logit (batch,)
        h = self.blocks(self.inp(x.transpose(1, 2)))
        return self.head(torch.cat([h.amax(-1), h.mean(-1)], 1)).squeeze(-1)

def bag_logits(logits, sizes):
    """MIL: điểm bag = trung bình top-k logit instance, k = max(1, ceil(n / 8))."""
    out = []
    for part in torch.split(logits, sizes.tolist()):
        k = max(1, math.ceil(len(part) / 8))
        out.append(part.topk(k).values.mean())
    return torch.stack(out)

def norm_stats(clips, max_inst=20000):
    idx = np.concatenate([INST_OF[c] for c in clips])
    idx = np.sort(np.random.default_rng(SEED).choice(idx, min(len(idx), max_inst), replace=False))
    f = feats_of(idx).reshape(-1, F.FEAT_DIM)
    return f.mean(0).astype(np.float32), np.maximum(f.std(0), 1e-3).astype(np.float32)

@torch.no_grad()
def predict(model, clips, mean, std):
    """Xác suất ĐÁNH NHAU mức clip; clip không có instance -> 0."""
    model.eval()
    probs = np.zeros(len(clips), np.float32)
    has = [i for i, c in enumerate(clips) if len(INST_OF[c])]
    ds = BagDS([clips[i] for i in has], False, mean, std)
    dl = torch.utils.data.DataLoader(ds, batch_size=32, collate_fn=collate, num_workers=0)
    out = []
    for x, sizes, _ in dl:
        out.append(torch.sigmoid(bag_logits(model(x.to(DEVICE)), sizes)).cpu())
    if has:
        probs[has] = torch.cat(out).numpy()
    return probs

def metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    auc = roc_auc_score(y, p) if len(set(y)) == 2 else float("nan")
    return dict(acc=accuracy_score(y, pred), f1=f1_score(y, pred, zero_division=0), auc=auc)

m = PairTCN()
print(f"PairTCN: {sum(p.numel() for p in m.parameters()) / 1e3:.0f}k tham so | thu forward:",
      m(torch.zeros(4, F.WINDOW, F.FEAT_DIM)).shape)
_kp, _box, _valid = augment(KP[:8], BOX[:8], VALID[:8], np.random.default_rng(0))
print("thu augment:", _kp.shape, _box.shape, _valid.shape)

## 7. Train + đánh giá: train RWF → test RLVS, train RLVS → test RWF, train gộp

In [ ]:
def run_experiment(tag, train_ds):
    sel = meta.ds.isin(train_ds)
    tr = meta[sel & (meta.sp == "train")]
    drop1 = int(((tr.y == 1) & ~tr.has_inst).sum())
    drop0 = int(((tr.y == 0) & ~tr.has_inst).sum())
    tr_clips = tr.index[tr.has_inst.values].tolist()
    va_clips = meta.index[sel & (meta.sp == "val")].tolist()
    print(f"\n===== {tag}: train tren {train_ds} | {len(tr_clips)} bag train "
          f"(bo {drop1} clip Fight + {drop0} clip NonFight khong co instance) | {len(va_clips)} clip val")
    mean, std = norm_stats(tr_clips)
    dl = torch.utils.data.DataLoader(BagDS(tr_clips, True, mean, std), batch_size=BATCH_BAGS, shuffle=True,
                                     collate_fn=collate, num_workers=N_WORKERS if N_WORKERS > 1 else 0,
                                     drop_last=len(tr_clips) > BATCH_BAGS, persistent_workers=N_WORKERS > 1)
    model = PairTCN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * max(1, len(dl)))
    loss_fn = nn.BCEWithLogitsLoss()
    best, ckpt = -1.0, os.path.join(WORK, f"best_{tag}.pt")
    yv = LABEL[va_clips]
    for ep in range(1, EPOCHS + 1):
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        for x, sizes, y in dl:
            logit = bag_logits(model(x.to(DEVICE)), sizes)
            loss = loss_fn(logit, y.to(DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            tot, nb = tot + loss.item(), nb + 1
        mv = metrics(yv, predict(model, va_clips, mean, std))
        mark = ""
        if mv["f1"] > best:
            best, mark = mv["f1"], "  *luu"
            torch.save({"model": model.state_dict(), "mean": mean, "std": std, "epoch": ep, "tag": tag,
                        "train_ds": train_ds, "val": mv}, ckpt)
        print(f"[{tag}] epoch {ep:2d}/{EPOCHS} loss {tot / max(nb, 1):.4f} | val acc {mv['acc']:.3f} "
              f"f1 {mv['f1']:.3f} auc {mv['auc']:.3f} | {time.time() - t0:.0f} s{mark}")
    ck = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    rows = []
    for test in ("RWF", "RLVS", "GOP"):
        ts = meta.index[(meta.sp == "val") & (meta.ds.isin(["RWF", "RLVS"] if test == "GOP" else [test]))].tolist()
        if not ts:
            continue
        p = predict(model, ts, ck["mean"], ck["std"])
        y = LABEL[ts]
        thr = np.linspace(0.05, 0.95, 19)
        f1s = [f1_score(y, (p >= t).astype(int), zero_division=0) for t in thr]
        ub = meta.loc[ts].query("y == 1").has_inst.mean()
        rows.append(dict(train=tag, test=f"{test} val", n_clip=len(ts), **metrics(y, p),
                         f1_tot_nhat=max(f1s), nguong_tot_nhat=thr[int(np.argmax(f1s))],
                         recall_toi_da=ub, best_epoch=ck["epoch"]))
    return model, ck, rows

RESULTS, MODELS = [], {}
for tag, train_ds in EXPERIMENTS:
    if not meta.ds.isin(train_ds).any():
        print("Bo qua", tag, "(khong co du lieu)")
        continue
    model, ck, rows = run_experiment(tag, train_ds)
    MODELS[tag] = (model, ck)
    RESULTS += rows
res = pd.DataFrame(RESULTS)
res.to_csv(os.path.join(WORK, "results.csv"), index=False)
print("\n===== KET QUA (muc clip, nguong 0.5; recall_toi_da = ti le clip Fight co >= 1 instance)")
display(res.round(3))

## 8. Xuất `pair_tcn.onnx` + `fight_config.json`

In [ ]:
model, ck = MODELS[EXPORT_FROM]
model = model.eval().cpu()
onnx_path = os.path.join(WORK, "pair_tcn.onnx")
dummy = torch.zeros(2, F.WINDOW, F.FEAT_DIM)
kw = dict(input_names=["x"], output_names=["logit"], opset_version=17,
          dynamic_axes={"x": {0: "batch"}, "logit": {0: "batch"}})
try:
    torch.onnx.export(model, dummy, onnx_path, dynamo=False, **kw)
except TypeError:                                    # torch cũ chưa có tham số dynamo
    torch.onnx.export(model, dummy, onnx_path, **kw)

cfg = {"fps": F.FPS, "window": F.WINDOW, "stride": F.STRIDE, "feat_dim": F.FEAT_DIM,
       "mean": ck["mean"].tolist(), "std": ck["std"].tolist(), "threshold": THRESHOLD,
       "trained_on": ck["train_ds"], "best_epoch": ck["epoch"], "val": {k: float(v) for k, v in ck["val"].items()},
       "input_note": "x = (pair_features - mean) / std; output logit theo instance, sigmoid -> xac suat"}
with open(os.path.join(WORK, "fight_config.json"), "w") as f:
    json.dump(cfg, f)

# kiểm tra ONNX cho cùng kết quả với PyTorch
idx = INST_OF[int(np.flatnonzero(meta.has_inst)[0])][:8]
x = ((feats_of(idx) - ck["mean"]) / ck["std"]).astype(np.float32)
ref = model(torch.from_numpy(x)).detach().numpy()
try:
    import onnxruntime as ort
    out = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"]).run(["logit"], {"x": x})[0]
    print(f"ONNX vs PyTorch: lech toi da {np.abs(out - ref).max():.2e} (batch {len(x)})")
except ImportError:
    print("(khong co onnxruntime tren Kaggle -> bo qua buoc so sanh; se kiem tra tren may)")
for fn in sorted(os.listdir(WORK)):
    print(f"{fn:28s} {os.path.getsize(os.path.join(WORK, fn)) / 1e6:8.2f} MB")